# 06 — Credit Line Adjustment & Final Output

- Rule-based credit line adjustment (segment + risk → percentage)
- XGBoost regressor for smoother, continuous adjustments
- Hard enforcement of `cu_line_incr_excl_flag`
- Clamping: min = balance + 10% buffer, max = 1.5× current limit
- SHAP explainability for XGBoost models
- **Final output table**: one row per account with all predictions

In [58]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import joblib
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import load_pipeline
from src import features as feat
from src.models import segmentation as seg
from src.models import risk_detector as rd
from src.models import credit_adjuster as ca
from src.models import spend_predictor as sp

pd.set_option('display.max_columns', 50)

In [59]:
con, counts = load_pipeline(verbose=True)

Loading CSV files into DuckDB…
  ✓ account_dim: 18,070 rows
  ✓ statement_fact: 658,228 rows
  ✓ transaction_fact: 493,336 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  ✓ wrld_stor_tran_fact: 1,053,854 rows
  ✓ syf_id: 18,070 rows
  ✓ rams_batch_cur: 96,799 rows
  ✓ fraud_claim_case: 77 rows
  ✓ fraud_claim_tran: 202 rows
  ✓ transaction_base (union): 1,547,190 rows

Verifying row counts…
  ✓ account_dim: 18,070 rows (OK)
  ✓ statement_fact: 658,228 rows (OK)
  ✓ transaction_fact: 493,336 rows (OK)
  ✓ wrld_stor_tran_fact: 1,053,854 rows (OK)
  ✓ transaction_base: 1,547,190 rows (OK)
  ✓ syf_id: 18,070 rows (OK)
  ✓ rams_batch_cur: 96,799 rows (OK)
  ✓ fraud_claim_case: 77 rows (OK)
  ✓ fraud_claim_tran: 202 rows (OK)


## 1. Rebuild Full Feature Pipeline

In [60]:
# Build customer base with all features
customer_base = feat.build_customer_base(con)
customer_monthly = feat.build_customer_monthly(con)
anomaly_features = feat.build_anomaly_features(con)

# Segmentation
customer_segmented = seg.assign_rule_based_segments(customer_base)

# Risk scoring
iso_model, iso_scaler, anomaly_preds = rd.train_isolation_forest(
    anomaly_features, fraud_rate_df=customer_base
)
customer_full = rd.compute_risk_score(customer_segmented, anomaly_preds)

print(f"Full pipeline data: {customer_full.shape}")
print(f"Segments: {customer_full['segment_name'].value_counts().to_dict()}")

  Calibrated contamination: 0.0085 (fraud rate: 77/18070)
  Detected 122 anomalous accounts (0.9%)
  Risk Score Distribution:
    Mean: 5.3
    Median: 0.8
    Std: 8.6
    Min: 0.0
    Max: 59.3
Full pipeline data: (18070, 65)
Segments: {'No Increase Needed': 12607, 'Eligible - No Risk': 3471, 'Eligible - With Risk': 1828, 'Non-Performing': 164}


## 2. Add Q4 Forecast

In [61]:
# Train XGBoost Q4 predictor
customer_monthly = feat.add_lag_features(customer_monthly)
customer_monthly = feat.add_spend_to_limit_ratio(customer_monthly, customer_base)
train_df, test_df, feature_cols, target_col = feat.build_q4_feature_matrix(
    customer_monthly, customer_base, cutoff_month='2024-10-01'
)

xgb_model, xgb_preds, xgb_metrics = sp.train_xgboost_spend(
    train_df, test_df, feature_cols, target_col
)

# Per-account Q4 forecast
q4_by_acct = test_df.copy()
q4_by_acct['predicted'] = xgb_preds
q4_forecast = q4_by_acct.groupby('current_account_nbr')['predicted'].sum().reset_index()
q4_forecast.columns = ['current_account_nbr', 'q4_forecast']

# Merge
customer_full = customer_full.merge(q4_forecast, on='current_account_nbr', how='left')
customer_full['q4_forecast'] = customer_full['q4_forecast'].fillna(0)

print(f"\nQ4 forecast coverage: {q4_forecast['current_account_nbr'].nunique()} accounts")

  XGBoost: MAE=$32.13  RMSE=$172.51  R²=0.9906

Q4 forecast coverage: 10771 accounts


## 3. Rule-Based Credit Line Adjustment

In [62]:
print("\nApplying rule-based adjustments...")
customer_adj = ca.apply_rule_based_adjustment(customer_full)

print(f"\nRule-based adjustment summary:")
print(customer_adj[['adjustment_pct', 'rule_recommended_line', 'rule_adjustment_delta']].describe().round(2))


Applying rule-based adjustments...

Rule-based adjustment summary:
       adjustment_pct  rule_recommended_line  rule_adjustment_delta
count        18070.00                18070.0                18070.0
mean             0.05                6641.97                 386.29
std              0.08                6603.63                 934.35
min             -0.05                    0.0               -2001.85
25%              0.00                 1560.0                    0.0
50%              0.00                 4500.0                    0.0
75%              0.10                 9600.0                  180.0
max              0.20                50000.0                 6000.0


In [63]:
# Check exclusion flag enforcement
excl = customer_adj[customer_adj['cu_line_incr_excl_flag'].astype(str).str.upper() == 'Y']
if len(excl) > 0:
    excl_increases = (excl['rule_adjustment_delta'] > 0).sum()
    print(f"\nExclusion flag accounts: {len(excl)}")
    print(f"  Accounts with increases despite flag: {excl_increases} (should be 0)")
else:
    print("\nNo accounts with exclusion flag set")


Exclusion flag accounts: 59
  Accounts with increases despite flag: 0 (should be 0)


## 4. XGBoost Credit Adjuster (Smooth Predictions)

In [64]:
print("\nTraining XGBoost credit adjuster...")
credit_model, credit_preds, credit_metrics, credit_features = ca.train_credit_adjuster(
    customer_adj,
    predicted_spend_col='q4_forecast',
    save_path='outputs/saved_models/xgb_credit_adjuster.joblib'
)


Training XGBoost credit adjuster...
  Credit Adjuster XGBoost: MAE=$17.68  RMSE=$62.47  R²=0.9954
  → Model saved to outputs/saved_models/xgb_credit_adjuster.joblib


## 5. Generate Final Recommendations

In [65]:
final_df = ca.generate_final_recommendations(
    customer_adj, credit_model, credit_features,
    predicted_spend_col='q4_forecast'
)

print(f"\nFinal recommendations generated for {len(final_df)} accounts")
print(f"\nAdjustment delta statistics:")
print(final_df['adjustment_delta'].describe().round(2))


Final recommendations generated for 18070 accounts

Adjustment delta statistics:
count    18070.0
mean      385.69
std       929.28
min      -1517.8
25%        -0.23
50%         0.33
75%       181.22
max      5882.63
Name: adjustment_delta, dtype: Float64


## 6. Credit Line Adjustment Waterfall

In [72]:
# Aggregate by segment
waterfall_data = final_df.groupby('segment_name').agg(
    current_total=('cu_crd_line', lambda x: pd.to_numeric(x, errors='coerce').sum()),
    recommended_total=('recommended_credit_line', 'sum'),
    delta_total=('adjustment_delta', 'sum'),
    account_count=('current_account_nbr', 'count')
).reset_index()

fig = go.Figure()

fig.add_trace(go.Bar(
    name='Current Credit Line',
    x=waterfall_data['segment_name'],
    y=waterfall_data['current_total'],
    marker_color='#3498db'
))

fig.add_trace(go.Bar(
    name='Recommended Credit Line',
    x=waterfall_data['segment_name'],
    y=waterfall_data['recommended_total'],
    marker_color='#2ecc71'
))

fig.add_trace(go.Bar(
    name='Adjustment Delta',
    x=waterfall_data['segment_name'],
    y=waterfall_data['delta_total'],
    marker_color='#e74c3c'
))

fig.update_layout(
    title='Credit Line Adjustment Waterfall — by Segment',
    xaxis_title='Segment',
    yaxis_title='Total Credit Line ($)',
    barmode='group',
    height=600, width=900,
    template='plotly_white'
)
fig.write_image("outputs/figures/credit_adjustment_waterfall.png", scale=2)
fig.show()

print("\nWaterfall Summary:")
print(waterfall_data.to_string(index=False))


Waterfall Summary:
        segment_name  current_total  recommended_total     delta_total  account_count
  Eligible - No Risk       30743189    36842640.934124  6099451.934124           3471
Eligible - With Risk        7943320     8797280.087622   853960.087622           1828
  No Increase Needed       73551781    73595410.592302    43629.592302          12607
      Non-Performing         801758      774177.320639   -27580.679361            164


## 7. SHAP Explainability

TreeExplainer on XGBoost spend predictor and credit adjuster.
- Summary beeswarm plot for global feature importance
- Waterfall plot for representative accounts (one per segment)

In [67]:
import shap

# --- SHAP for Q4 Spend Predictor ---
print("Computing SHAP values for Q4 Spend Predictor...")
explainer_spend = shap.TreeExplainer(xgb_model)

# Use a sample for speed
sample_size = min(2000, len(test_df))
X_sample = test_df[feature_cols].astype(float).sample(sample_size, random_state=42)
shap_values_spend = explainer_spend.shap_values(X_sample)

# Beeswarm plot
fig_shap = plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values_spend, X_sample, show=False, max_display=15)
plt.title('SHAP — Q4 Spend Predictor Feature Importance')
plt.tight_layout()
plt.savefig('outputs/figures/shap_spend_predictor.png', dpi=200, bbox_inches='tight')
plt.close()
print("  → Saved SHAP beeswarm for Q4 predictor")

Computing SHAP values for Q4 Spend Predictor...
  → Saved SHAP beeswarm for Q4 predictor


In [73]:
# --- SHAP for Credit Adjuster ---
print("\nComputing SHAP values for Credit Adjuster...")
explainer_credit = shap.TreeExplainer(credit_model)

credit_feat_cols = [c for c in credit_features if c in customer_adj.columns]
X_credit_sample = customer_adj[credit_feat_cols].fillna(0).astype(float).sample(
    min(2000, len(customer_adj)), random_state=42
)
shap_values_credit = explainer_credit.shap_values(X_credit_sample)

fig_shap2 = plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values_credit, X_credit_sample, show=False, max_display=15)
plt.title('SHAP — Credit Line Adjuster Feature Importance')
plt.tight_layout()
plt.savefig('outputs/figures/shap_credit_adjuster.png', dpi=200, bbox_inches='tight')
plt.close()
print("  → Saved SHAP beeswarm for credit adjuster")


Computing SHAP values for Credit Adjuster...
  → Saved SHAP beeswarm for credit adjuster


In [69]:
# Waterfall plots for representative accounts (one per segment)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

for seg_label, seg_name in seg.SEGMENTS.items():
    seg_accounts = final_df[final_df['segment_name'] == seg_name]
    if len(seg_accounts) == 0:
        continue

    # Pick the median-adjustment account
    acct_id = seg_accounts.iloc[len(seg_accounts)//2]['current_account_nbr']

    # Find this account in credit features
    acct_row = customer_adj[customer_adj['current_account_nbr'] == acct_id]
    if len(acct_row) == 0:
        continue

    X_acct = acct_row[credit_feat_cols].fillna(0).astype(float)
    sv = explainer_credit.shap_values(X_acct)
    explanation = shap.Explanation(
        values=sv[0],
        base_values=explainer_credit.expected_value,
        data=X_acct.values[0],
        feature_names=credit_feat_cols
    )

    fig_w = plt.figure(figsize=(10, 6))
    shap.waterfall_plot(explanation, show=False, max_display=12)
    plt.title(f'SHAP Waterfall — {seg_name} (Account: {acct_id[:8]}…)')
    plt.tight_layout()
    plt.savefig(f'outputs/figures/shap_waterfall_{seg_label}.png', dpi=200, bbox_inches='tight')
    plt.close()
    print(f"  → Saved waterfall for {seg_name}")

  → Saved waterfall for Non-Performing
  → Saved waterfall for No Increase Needed
  → Saved waterfall for Eligible - With Risk
  → Saved waterfall for Eligible - No Risk


## 8. Monthly Spend Heatmap

In [70]:
# Build heatmap: accounts × months
pivot_data = customer_monthly.copy()
pivot_data['month_str'] = pivot_data['month'].dt.strftime('%Y-%m')

# Sample top 200 accounts by total spend for readability
top_accounts = pivot_data.groupby('current_account_nbr')['total_spend'].sum()\
    .nlargest(200).index

heatmap_df = pivot_data[pivot_data['current_account_nbr'].isin(top_accounts)]\
    .pivot_table(values='total_spend', index='current_account_nbr', 
                 columns='month_str', fill_value=0)

fig = px.imshow(
    heatmap_df.values,
    labels=dict(x="Month", y="Account", color="Spend ($)"),
    x=heatmap_df.columns.tolist(),
    color_continuous_scale='YlOrRd',
    title='Monthly Spend Heatmap — Top 200 Accounts by Total Spend',
    aspect='auto'
)
fig.update_layout(height=800, width=1000)
fig.write_image("outputs/figures/monthly_spend_heatmap.png", scale=2)
fig.show()

## 9. Build Final Output Table

In [71]:
output = final_df[[
    'current_account_nbr', 'cu_crd_line', 'q4_forecast',
    'segment_name', 'risk_score', 'anomaly_flag',
    'recommended_credit_line', 'adjustment_delta'
]].copy()

output = output.rename(columns={'cu_crd_line': 'current_credit_line'})
output = output.sort_values('adjustment_delta', ascending=False)

output.to_csv('outputs/predictions/final_output.csv', index=False)

print(f"\n{'='*60}")
print(f"  FINAL OUTPUT TABLE")
print(f"{'='*60}")
print(f"  Rows: {len(output):,}")
print(f"  Columns: {output.columns.tolist()}")
print(f"\n  Top 10 by adjustment delta:")
print(output.head(10).to_string(index=False))
print(f"\n  Bottom 5 (reductions):")
print(output.tail(5).to_string(index=False))


  FINAL OUTPUT TABLE
  Rows: 18,070
  Columns: ['current_account_nbr', 'current_credit_line', 'q4_forecast', 'segment_name', 'risk_score', 'anomaly_flag', 'recommended_credit_line', 'adjustment_delta']

  Top 10 by adjustment delta:
current_account_nbr  current_credit_line  q4_forecast       segment_name  risk_score  anomaly_flag  recommended_credit_line  adjustment_delta
   TAzeyite6WhlcCR1                30000     0.000000 Eligible - No Risk        0.25           0.0             35882.634766       5882.634766
   0IvfHIuwjMYPujyc                30000     0.000000 Eligible - No Risk        0.25           0.0             35875.983887       5875.983887
   6l0K4TCrnGviATbJ                30000  1405.959595 Eligible - No Risk         0.0           0.0             35641.667969       5641.667969
   wgGAH69Bdkx3Yzak                30000  5548.342285 Eligible - No Risk         0.0           0.0             35290.963867       5290.963867
   StOw8eiBY4n4hBKu                30000  6027.016113 El

## Summary

| Output | Description |
|--------|-------------|
| `final_output.csv` | One row per account: current line, Q4 forecast, segment, risk, recommendation |
| Models saved | XGBoost spend predictor, RF segmenter, Isolation Forest, XGBoost credit adjuster |
| SHAP plots | Beeswarm + waterfall for spend predictor and credit adjuster |
| All 7 visualizations | Heatmap, segment dist, forecast scatter, risk by segment, waterfall, SHAP, anomaly |